In [1]:
import warnings ;warnings.filterwarnings('ignore');import sys ;sys.path.append("../../")
from  CommonFunc_04 import *;DataPreprocessing.plotSetting(setting_info=False)
import tensorflow as tf, numpy as np
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
print(yellow(f"tensorflow version : {tf.__version__}"))



  - ◎ matplot graph set complete
tensorflow version : 2.17.0


---
#  chapter 05. 자연어 처리 소개

## 5.1 언어를 숫자로 인코딩
- antigram : 어떤 단어의 아나그램이지만 반대 뜻을 가진 단어. 는 글자를 숫자로 인코딩하는게 어렵게 만듬. 
-  단어자체에 숫자를 부여하는 것이 더 좋음. -> 토큰화 

### 5.1.1. 토큰화

In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer

sentences  =[
    'Today is a sunny day',
    'Today is a rainy day'
]

tokenizer = Tokenizer(num_words = 100) # 객체 생성, 토큰화가능 개수 지정
tokenizer.fit_on_texts(sentences)
word_index =tokenizer.word_index
print(word_index) ## vocabulary

{'today': 1, 'is': 2, 'a': 3, 'day': 4, 'sunny': 5, 'rainy': 6}


> 말뭉치(corpus)에서 추출할수 있는 최대 토큰 개수   
> word index 를 vocabulary 라고도 부름.     
> index_word 속성은 반대로 인덱스와 단어의 키/값 쌍으로 구성됨.     

### 5.1.2. 문장을 시퀀스로 바꾸기  

In [3]:

a = ['Is it sunny today?']
if a in sentences:
    print()
else :
    sentences.extend(a)
print(sentences)
tokenizer = Tokenizer(num_words=100)
tokenizer.fit_on_texts(sentences)
word_index = tokenizer.word_index
sequences = tokenizer.texts_to_sequences(sentences) ## text 를 토큰의 나열로 변환
print(sequences)

['Today is a sunny day', 'Today is a rainy day', 'Is it sunny today?']
[[1, 2, 3, 4, 5], [1, 2, 3, 6, 5], [2, 7, 4, 1]]


> OOV 토큰 사용하기 (out-of-vocabulary)

In [4]:
test_data =[ 'Today is a snowy day', 'Will it be rainy tomorrow?']
test_sequences = tokenizer.texts_to_sequences(test_data)
print(word_index)
print(test_sequences)
print(tokenizer.sequences_to_texts(test_sequences))

{'today': 1, 'is': 2, 'a': 3, 'sunny': 4, 'day': 5, 'rainy': 6, 'it': 7}
[[1, 2, 3, 5], [7, 6]]
['today is a day', 'it rainy']


In [5]:
tokenizer = Tokenizer(num_words=100, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)
word_index = tokenizer.word_index
test_sequences= tokenizer.texts_to_sequences(test_data)
print(test_sequences)
print(tokenizer.sequences_to_texts(test_sequences))

[[2, 3, 4, 1, 6], [1, 8, 1, 7, 1]]
['today is a <OOV> day', '<OOV> it <OOV> rainy <OOV>']


> 최신 텐서플로 layer 에서 제공하는 토크나이저 사용

In [6]:
tv = keras.layers.TextVectorization()
tv.adapt(sentences)
print(tv.get_vocabulary())


['', '[UNK]', 'today', 'is', 'sunny', 'day', 'a', 'rainy', 'it']


In [7]:
test_seq = tv(test_data)
print(test_seq)
print(test_seq.numpy())

tf.Tensor(
[[2 3 6 1 5]
 [1 8 1 7 1]], shape=(2, 5), dtype=int64)
[[2 3 6 1 5]
 [1 8 1 7 1]]


> 패딩 이해하기 

In [36]:
a='I really enjoyed walking in the snow today'
sentences.extend([a])if a not in sentences else print()
from pprint import pprint;pprint(sentences)
tokenizer.fit_on_texts(sentences)
sequences =tokenizer.texts_to_sequences(sentences)
pprint(sequences)
from tensorflow.keras.preprocessing.sequence import pad_sequences
padded =pad_sequences(sequences)
print(padded) ## 기본값은 prepadding 
padded = pad_sequences(sequences, padding='post')
print(padded)
## 최대 길이 맞춰 패딩 
padded = pad_sequences(sequences, padding='post', maxlen=6 )
print(padded)
padded = pad_sequences(sequences, padding='post', maxlen=6, truncating='post' )
print(padded)


['Today is a sunny day',
 'Today is a rainy day',
 'Is it sunny today?',
 'I really enjoyed walking in the snow today']
[[2, 3, 4, 5, 6], [2, 3, 4, 7, 6], [3, 8, 5, 2], [9, 10, 11, 12, 13, 14, 15, 2]]
[[ 0  0  0  2  3  4  5  6]
 [ 0  0  0  2  3  4  7  6]
 [ 0  0  0  0  3  8  5  2]
 [ 9 10 11 12 13 14 15  2]]
[[ 2  3  4  5  6  0  0  0]
 [ 2  3  4  7  6  0  0  0]
 [ 3  8  5  2  0  0  0  0]
 [ 9 10 11 12 13 14 15  2]]
[[ 2  3  4  5  6  0]
 [ 2  3  4  7  6  0]
 [ 3  8  5  2  0  0]
 [11 12 13 14 15  2]]
[[ 2  3  4  5  6  0]
 [ 2  3  4  7  6  0]
 [ 3  8  5  2  0  0]
 [ 9 10 11 12 13 14]]


## 5.2 불용어 제거와 텍스트 정제
https://github.com/rickiepark/aiml4coders/tree/main/ch05/05-intro-nlp.ipynb
<!-- # from bs4 import BeautifulSoup
# soup = BeautifulSoup(sentence)
# sentence = soup.get_text() -->

In [37]:
stopwords = ["a", "about", "above", "after", "again", "against", "all", "am", "an", "and", "any", "are", "as", "at",
            "be", "because", "been", "before", "being", "below", "between", "both", "but", "by", "could", "did", "do",
            "does", "doing", "down", "during", "each", "few", "for", "from", "further", "had", "has", "have", "having",
            "he", "hed", "hes", "her", "here", "heres", "hers", "herself", "him", "himself", "his", "how",
            "hows", "i", "id", "ill", "im", "ive", "if", "in", "into", "is", "it", "its", "itself",
            "lets", "me", "more", "most", "my", "myself", "nor", "of", "on", "once", "only", "or", "other", "ought",
            "our", "ours", "ourselves", "out", "over", "own", "same", "she", "shed", "shell", "shes", "should",
            "so", "some", "such", "than", "that", "thats", "the", "their", "theirs", "them", "themselves", "then",
            "there", "theres", "these", "they", "theyd", "theyll", "theyre", "theyve", "this", "those", "through",
            "to", "too", "under", "until", "up", "very", "was", "we", "wed", "well", "were", "weve", "were",
            "what", "whats", "when", "whens", "where", "wheres", "which", "while", "who", "whos", "whom", "why",
            "whys", "with", "would", "you", "youd", "youll", "youre", "youve", "your", "yours", "yourself",
            "yourselves"]





## 5.3  실제 데이터 다루기 
- tfds 에서 IMdb( internet movie database) 를 다운받아서 다루어보자 


In [40]:
import tensorflow_datasets as tfds
imdb_sentences = []
train_data = tfds.as_numpy(tfds.load('imdb_reviews', split  ='train'))



2024-10-18 13:18:14.426448: W external/local_tsl/tsl/platform/cloud/google_auth_provider.cc:184] All attempts to get a Google authentication bearer token failed, returning an empty token. Retrieving token from files failed with "NOT_FOUND: Could not locate the credentials file.". Retrieving token from GCE failed with "FAILED_PRECONDITION: Error executing an HTTP request: libcurl code 6 meaning 'Couldn't resolve host name', error details: Could not resolve host: metadata.google.internal".


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...:   0%|          | 0/25000 [00:00<?, ? examples/s]

...

Generating test examples...:   0%|          | 0/25000 [00:00<?, ? examples/s]

Shuffling /Users/forrestdpark/tensorflow_datasets/imdb_reviews/plain_text/incomplete.VPEEZ1_1.0.0/imdb_reviews…

Generating unsupervised examples...:   0%|          | 0/50000 [00:00<?, ? examples/s]

Shuffling /Users/forrestdpark/tensorflow_datasets/imdb_reviews/plain_text/incomplete.VPEEZ1_1.0.0/imdb_reviews…

Dataset imdb_reviews downloaded and prepared to /Users/forrestdpark/tensorflow_datasets/imdb_reviews/plain_text/1.0.0. Subsequent calls will reuse this data.


In [51]:
for item in train_data:
    imdb_sentences.append(str(item['text']))

tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words =5000)
tokenizer.fit_on_texts(imdb_sentences)
sequences = tokenizer.texts_to_sequences(imdb_sentences)


    
    

2024-10-18 13:37:15.644258: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


the
and
a
of
to
is
br
in
it
i


> < br >등의 html 용어 삭제

In [54]:

import string
from bs4 import BeautifulSoup
table = str.maketrans('','', string.punctuation)

imdb_sentences =[]
train_data = tfds.as_numpy(tfds.load('imdb_reviews',split='train'))
for item in train_data:
    sentence = str(item['text'].decode('UTF-8').lower())
    soup = BeautifulSoup(sentence)
    sentence = soup.get_text()
    words = sentence.split()
    filtered_sentence = ""
    for word in words :
        word = word.translate(table)
        if word not in stopwords:
            filtered_sentence = filtered_sentence+ word + " "
    imdb_sentences.append(filtered_sentence)
tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words =25000)
tokenizer.fit_on_texts(imdb_sentences)
sequences = tokenizer.texts_to_sequences(imdb_sentences)

2024-10-18 13:42:54.985418: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


absolutely
terrible
movie
dont
lured
christopher
walken
michael
ironside
great


In [60]:
for order,i in enumerate(tokenizer.word_index):
    if order< 10:
        print(order+1,i)

1 movie
2 film
3 not
4 one
5 like
6 just
7 good
8 even
9 no
10 time


> 단어이음  변환

In [61]:
## him/her -> him / her 두단어로 토큰화 됨

sentence = sentence.replace(","," , ")
sentence = sentence.replace("."," . ")
sentence = sentence.replace("-"," - ")
sentence = sentence.replace("/"," / ")



In [64]:
sentences =[
    'Today is a sunny day',
    'Today is a rainy day',
    'Is it sunny today?'
]
sequences = tokenizer.texts_to_sequences(sentences)
print(sequences)
print(tokenizer.sequences_to_texts(sequences))

[[524, 5100, 157], [524, 6288, 157], [5100, 524]]
['today sunny day', 'today rainy day', 'sunny today']
